# 07 — Pipeline temporal e integración del prototipo

Este notebook avanza la arquitectura mientras se completa el conjunto de imágenes y se recopilan videos. La rama temporal se prepara para reconocer **J**, **Z** y **otro movimiento/no J-Z** mediante flujo óptico y una CNN 3D. Al final se dejan utilidades para combinar esa salida con la clasificación estática y acumular letras.

## Estado y alcance

- La extracción de mapas y el entrenamiento estático se realizan en el notebook 06. Este notebook no depende de que esa extracción haya terminado para preparar el código de video.
- La rama temporal **no se puede entrenar ni validar** hasta tener videos etiquetados de J, Z y ejemplos de otros movimientos/no J-Z.
- El diseño de flujo óptico usa una región fija para todo el clip, calculada con las detecciones MediaPipe de la secuencia. Así se conserva el desplazamiento de la mano entre fotogramas.
- Esto es un prototipo de deletreo de letras, no traducción completa de ASL. La captura en vivo y la interfaz de cámara se integrarán después de evaluar los modelos con videos.

## Estructura esperada para videos

Guarda los clips en Drive así; incluye identificador de participante para poder separar entrenamiento y prueba por persona:

```text
data/videos_jz/
├── J/<participante>/<clip>.mp4
├── Z/<participante>/<clip>.mp4
└── other/<participante>/<clip>.mp4
```

`other` debe incluir clips donde no se está haciendo J ni Z (mano quieta, otras señas o transiciones). Tener solo clips J/Z obligaría al sistema a escoger una de esas dos clases siempre.

In [ ]:
# En Colab: instala MediaPipe si el notebook 06 no dejó la dependencia en este entorno.
# Si ya está instalado, esta celda no cambia nada.
%pip install -q mediapipe==0.10.35 scikit-learn

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import csv
import json
import urllib.request
import warnings

import cv2
import mediapipe as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path("/content/drive/MyDrive/proyecto-ia2")
VIDEO_ROOT = PROJECT_ROOT / "data/videos_jz"
TEMPORAL_DIR = PROJECT_ROOT / "data/processed/video_flow"
MANIFEST_PATH = TEMPORAL_DIR / "video_manifest.csv"
MODEL_ASSET = PROJECT_ROOT / "data/processed/landmark_map_experiment/hand_landmarker.task"

# Parámetros compartidos entre extracción y modelo temporal.
SEQUENCE_FRAMES = 17        # produce 16 campos de flujo óptico
FLOW_SIZE = 128
FLOW_CLIP_PIXELS = 20.0
BATCH_SIZE = 8
TEMPORAL_MODEL_PATH = TEMPORAL_DIR / "jz_flow_cnn.keras"

TEMPORAL_DIR.mkdir(parents=True, exist_ok=True)
if not MODEL_ASSET.exists():
    MODEL_ASSET.parent.mkdir(parents=True, exist_ok=True)
    model_url = (
        "https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
        "hand_landmarker/float16/latest/hand_landmarker.task"
    )
    urllib.request.urlretrieve(model_url, MODEL_ASSET)

print("Videos:", VIDEO_ROOT)
print("Características y checkpoints:", TEMPORAL_DIR)
print("MediaPipe:", MODEL_ASSET.exists())

## 1. Preparar flujo óptico por clip

Para cada video se eligen 17 fotogramas uniformemente espaciados. MediaPipe detecta la mano en esos fotogramas y se calcula una región cuadrada común a toda la secuencia. Después se calcula flujo óptico denso entre pares consecutivos y se normalizan sus componentes horizontal y vertical a `[-1, 1]`.

El manifiesto se actualiza clip por clip. Si la sesión se interrumpe, se puede volver a ejecutar la preparación y reutilizar los archivos que ya aparecen en el manifiesto. Revisa algunos clips y la tasa de detección antes de procesar una colección grande.

In [ ]:
def sample_video_frames(video_path, count=SEQUENCE_FRAMES):
    cap = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened() or total < 2:
        cap.release()
        return None
    indices = np.linspace(0, total - 1, count).round().astype(int)
    frames = []
    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx))
        ok, frame = cap.read()
        if not ok:
            cap.release()
            return None
        frames.append(frame)
    cap.release()
    return frames


def detected_points(frame_bgr, detector):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    result = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not result.hand_landmarks:
        return None
    h, w = frame_bgr.shape[:2]
    return np.asarray([(p.x * w, p.y * h) for p in result.hand_landmarks[0]], dtype=np.float32)


def fixed_hand_crops(frames, points_per_frame, padding=0.35):
    """Usa una sola ROI para todo el clip para conservar el movimiento global de la mano."""
    all_points = np.concatenate(points_per_frame, axis=0)
    x_min, y_min = all_points.min(axis=0)
    x_max, y_max = all_points.max(axis=0)
    cx, cy = (x_min + x_max) / 2, (y_min + y_max) / 2
    side = max(x_max - x_min, y_max - y_min) * (1 + 2 * padding)
    side = max(float(side), 32.0)
    x0, y0 = int(np.floor(cx - side / 2)), int(np.floor(cy - side / 2))
    side = int(np.ceil(side))
    crops = []
    for frame in frames:
        h, w = frame.shape[:2]
        x1, y1 = x0 + side, y0 + side
        left, top = max(0, -x0), max(0, -y0)
        right, bottom = max(0, x1 - w), max(0, y1 - h)
        padded = cv2.copyMakeBorder(frame, top, bottom, left, right, cv2.BORDER_CONSTANT, value=(0, 0, 0))
        crop = padded[y0 + top:y1 + top, x0 + left:x1 + left]
        crops.append(cv2.resize(crop, (FLOW_SIZE, FLOW_SIZE), interpolation=cv2.INTER_AREA))
    return crops


def video_to_flow(video_path, detector):
    frames = sample_video_frames(video_path)
    if frames is None:
        return None, "no se pudo leer el video"
    points = [detected_points(frame, detector) for frame in frames]
    if any(p is None for p in points):
        return None, "MediaPipe no detectó mano en todos los fotogramas muestreados"
    crops = fixed_hand_crops(frames, points)
    grays = [cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY) for crop in crops]
    flows = []
    for prev_gray, next_gray in zip(grays[:-1], grays[1:]):
        flow = cv2.calcOpticalFlowFarneback(
            prev_gray, next_gray, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
        )
        flows.append(np.clip(flow / FLOW_CLIP_PIXELS, -1.0, 1.0).astype(np.float32))
    return np.stack(flows), ""


def canonical_video_label(folder_name):
    key = folder_name.strip().lower()
    aliases = {"j": "J", "z": "Z", "other": "other", "none": "other", "background": "other"}
    return aliases.get(key)


def prepare_video_dataset():
    if not VIDEO_ROOT.is_dir():
        raise FileNotFoundError(f"Crea primero la estructura de videos en {VIDEO_ROOT}")
    existing = {}
    if MANIFEST_PATH.exists():
        old = pd.read_csv(MANIFEST_PATH).drop_duplicates("video_path", keep="last")
        existing = {str(row.video_path): row for row in old.itertuples(index=False)}
    video_paths = sorted(p for p in VIDEO_ROOT.rglob("*") if p.suffix.lower() in {".mp4", ".mov", ".avi", ".mkv"})
    if not video_paths:
        print("Aún no hay videos. Sube clips J, Z y other a VIDEO_ROOT y vuelve a ejecutar esta celda.")
        return pd.DataFrame()
    write_header = not MANIFEST_PATH.exists() or MANIFEST_PATH.stat().st_size == 0
    detector_options = mp.tasks.vision.HandLandmarkerOptions(
        base_options=mp.tasks.BaseOptions(model_asset_path=str(MODEL_ASSET)),
        running_mode=mp.tasks.vision.RunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.5,
        min_hand_presence_confidence=0.5,
    )
    rows = []
    detector = mp.tasks.vision.HandLandmarker.create_from_options(detector_options)
    try:
        with MANIFEST_PATH.open("a", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=["video_path", "label", "signer", "flow_path", "ok", "error"])
            if write_header:
                writer.writeheader()
            for video_path in video_paths:
                rel = video_path.relative_to(VIDEO_ROOT).as_posix()
                label = canonical_video_label(video_path.relative_to(VIDEO_ROOT).parts[0])
                if label is None:
                    warnings.warn(f"Ignoro carpeta/clase no reconocida: {rel}")
                    continue
                parts = video_path.relative_to(VIDEO_ROOT).parts
                signer = parts[1] if len(parts) > 2 else "unknown"
                output = TEMPORAL_DIR / "flows" / label / signer / f"{video_path.stem}.npy"
                previous = existing.get(rel)
                if previous and output.is_file() and int(previous.ok) == 1:
                    rows.append(previous._asdict())
                    continue
                output.parent.mkdir(parents=True, exist_ok=True)
                flow, error = video_to_flow(video_path, detector)
                ok = flow is not None
                if ok:
                    np.save(output, flow)
                record = {"video_path": rel, "label": label, "signer": signer,
                          "flow_path": output.relative_to(TEMPORAL_DIR).as_posix(),
                          "ok": int(ok), "error": error}
                writer.writerow(record)
                stream.flush()
                rows.append(record)
                print(f"{len(rows)}/{len(video_paths)} | {rel} | {'OK' if ok else error}")
    finally:
        detector.close()
    return pd.read_csv(MANIFEST_PATH).drop_duplicates("video_path", keep="last")

# Ejecuta cuando haya videos; no es necesario detener el procesamiento de imágenes del notebook 06.
# video_df = prepare_video_dataset()


## 2. CNN temporal sobre flujo óptico

La entrada de la red es una secuencia de 16 imágenes de flujo con dos canales (`x`, `y`). Una CNN 3D aprende patrones espaciales y su evolución temporal. La división se hace agrupando por participante para evitar que clips de una misma persona aparezcan en entrenamiento y prueba.

No ejecutes el entrenamiento hasta tener suficientes videos de varias personas para las tres clases. El bloque de abajo queda preparado; primero revisa el manifiesto y el balance de clases.

In [ ]:
FLOW_SHAPE = (SEQUENCE_FRAMES - 1, FLOW_SIZE, FLOW_SIZE, 2)
TEMPORAL_CLASSES = ["J", "Z", "other"]


def build_flow_cnn(num_classes=len(TEMPORAL_CLASSES)):
    inputs = tf.keras.Input(shape=FLOW_SHAPE, name="optical_flow_sequence")
    x = tf.keras.layers.Conv3D(16, (3, 5, 5), strides=(1, 2, 2), padding="same", activation="relu")(inputs)
    x = tf.keras.layers.MaxPool3D((1, 2, 2))(x)
    x = tf.keras.layers.Conv3D(32, (3, 3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPool3D((2, 2, 2))(x)
    x = tf.keras.layers.Conv3D(64, (3, 3, 3), padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling3D()(x)
    x = tf.keras.layers.Dropout(0.35)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs, outputs, name="jz_optical_flow_cnn")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


def load_flow_table(manifest):
    usable = manifest[manifest.ok.astype(int) == 1].copy()
    if not set(TEMPORAL_CLASSES).issubset(set(usable.label)):
        raise ValueError(f"Se necesitan ejemplos de las tres clases: {TEMPORAL_CLASSES}")
    usable["label_id"] = usable.label.map({name: i for i, name in enumerate(TEMPORAL_CLASSES)})
    X = np.stack([np.load(TEMPORAL_DIR / p) for p in usable.flow_path])
    y = usable.label_id.to_numpy(dtype=np.int32)
    groups = usable.signer.to_numpy()
    if len(np.unique(groups)) < 3:
        raise ValueError("Se necesitan al menos tres participantes para train/validación/test por persona.")
    return X, y, groups, usable


def split_by_signer(X, y, groups, seed=SEED):
    first = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=seed)
    train_idx, hold_idx = next(first.split(X, y, groups))
    second = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=seed + 1)
    val_rel, test_rel = next(second.split(X[hold_idx], y[hold_idx], groups[hold_idx]))
    return train_idx, hold_idx[val_rel], hold_idx[test_rel]

# Cuando el manifiesto contenga varias personas y las tres clases:
# video_df = prepare_video_dataset()
# X_flow, y_flow, signer_ids, flow_rows = load_flow_table(video_df)
# train_i, val_i, test_i = split_by_signer(X_flow, y_flow, signer_ids)
# flow_model = build_flow_cnn()
# flow_model.fit(
#     X_flow[train_i], y_flow[train_i], validation_data=(X_flow[val_i], y_flow[val_i]),
#     epochs=30, batch_size=BATCH_SIZE,
#     callbacks=[
#         tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True),
#         tf.keras.callbacks.BackupAndRestore(backup_dir=str(TEMPORAL_DIR / "backup_flow_cnn")),
#         tf.keras.callbacks.ModelCheckpoint(str(TEMPORAL_DIR / "best_flow_cnn.keras"),
#                                            monitor="val_loss", save_best_only=True),
#     ], verbose=2,
# )
# flow_model.save(TEMPORAL_MODEL_PATH)
# y_hat = flow_model.predict(X_flow[test_i]).argmax(axis=1)
# print(classification_report(y_flow[test_i], y_hat, target_names=TEMPORAL_CLASSES, zero_division=0))
# print("Macro F1:", f1_score(y_flow[test_i], y_hat, average="macro"))


## 3. Combinación de ramas y texto

La rama estática devuelve probabilidades de letras estáticas; la dinámica devuelve `J`, `Z` o `other`. La regla inicial da prioridad a J/Z solo cuando la rama temporal supera su umbral; si no, usa la salida estática. Los umbrales deben calibrarse con validación, no elegirse mirando el test.

La clase `LetterBuffer` es solo la lógica de interfaz: start/stop, añadir letra, espacio y borrar. La detección de cuándo una seña termina y cuándo se acepta una letra debe ajustarse con videos reales; esta celda no la inventa ni afirma que ya reconoce en vivo.

In [ ]:
def combine_predictions(static_label, static_confidence,
                        dynamic_label, dynamic_confidence,
                        dynamic_threshold=0.80):
    """Combina salidas; dynamic_label es J, Z u other."""
    if dynamic_label in {"J", "Z"} and dynamic_confidence >= dynamic_threshold:
        return dynamic_label, float(dynamic_confidence), "dynamic"
    if static_label is None:
        return None, 0.0, "uncertain"
    return static_label, float(static_confidence), "static"


class LetterBuffer:
    """Buffer de texto simple controlado por eventos de la interfaz."""
    def __init__(self):
        self.active = False
        self.text = ""

    def start(self):
        self.active = True

    def stop(self):
        self.active = False
        return self.text

    def append_letter(self, letter):
        if self.active and isinstance(letter, str) and len(letter) == 1 and letter.isalpha():
            self.text += letter.upper()

    def add_space(self):
        if self.active and self.text and not self.text.endswith(" "):
            self.text += " "

    def backspace(self):
        if self.active:
            self.text = self.text[:-1]

    def clear(self):
        self.text = ""

# Ejemplo aislado de la interfaz; no invoca modelos ni cámara.
buffer = LetterBuffer()
buffer.start()
buffer.append_letter("H")
buffer.append_letter("O")
buffer.append_letter("L")
buffer.append_letter("A")
print("Vista previa:", buffer.text)
print("Texto al detener:", buffer.stop())

## Criterios antes de conectar la cámara

1. Revisar que el preprocesamiento detecte correctamente la mano y que el flujo óptico muestre movimiento de la mano, no solo ruido del fondo.
2. Entrenar y evaluar J/Z por participante; informar accuracy, macro F1, matriz de confusión y latencia por clip.
3. Cargar el mejor modelo estático de notebook 06 y confirmar que su preprocesamiento de mapas sea idéntico al de entrenamiento.
4. Implementar captura en vivo. En Colab la cámara del navegador requiere una integración JavaScript; `cv2.VideoCapture(0)` normalmente no accede directamente a la cámara del computador desde un runtime alojado.
5. Añadir estabilización/segmentación temporal y probar palabras completas. La corrección ortográfica debe ser una sugerencia posterior, no sustituir silenciosamente las letras observadas.